# Example of SasRec training/inference

This example uses `ParquetModule` for data loading and processing. 

**To improve performance, it requires setting the environment variables ARROW_NUM_THREADS and ARROW_IO_THREADS to match the number of available CPUs.**

**Then, the environment variables OMP_NUM_THREADS and MKL_NUM_THREADS and torch varibles num_threads and num_interop_threads should be set to 1.**

In [1]:
%env OMP_NUM_THREADS=1
%env MKL_NUM_THREADS=1
%env ARROW_NUM_THREADS=8x
%env ARROW_IO_THREADS=8

env: OMP_NUM_THREADS=1
env: MKL_NUM_THREADS=1
env: ARROW_NUM_THREADS=8x
env: ARROW_IO_THREADS=8


In [2]:
import torch
import lightning as L
import pandas as pd

torch.set_num_threads(1)
torch.set_num_interop_threads(1)

L.seed_everything(42);

Seed set to 42


## Preparing data
In this example, we will use the MovieLens dataset, namely the 1M subset. A simple case is demonstrated, so only item ids will be used as a model input.

Note: Current implementation of SasRec is able to handle item and interactions features. It does not take into account user features. 

In [3]:
interactions = pd.read_csv("/home/jovyan/Pembek/RePlay/examples/data/ml1m_ratings.dat", sep="\t", names=["user_id", "item_id","rating","timestamp"])
interactions = interactions.drop(columns=["rating"])

In [4]:
interactions["timestamp"] = interactions["timestamp"].astype("int64")
interactions = interactions.sort_values(by="timestamp")
interactions["timestamp"] = interactions.groupby("user_id").cumcount()
interactions

,user_id,item_id,timestamp
1000138,6040,858,0
999873,6040,593,1
1000153,6040,2384,2
1000007,6040,1961,3
1000192,6040,2019,4
...,...,...,...
825793,4958,2399,446
825438,4958,1407,447
825731,4958,2634,448
825724,4958,3264,449


### Encode catagorical data
To ensure all categorical data is fit for training, it needs to be encoded using the `LabelEncoder` class. Create an instance of the encoder, providing a `LabelEncodingRule` for each categorical column in the dataset that will be used in a model. Note that ids of users and ids of items are always used.

In [5]:
from replay.preprocessing import LabelEncoder, LabelEncodingRule

encoder = LabelEncoder(
    [
        LabelEncodingRule("user_id"),
        LabelEncodingRule("item_id"),
    ]
)
encoded_interactions = encoder.fit_transform(interactions)
encoded_interactions

,timestamp,user_id,item_id
0,0,6039,802
1,1,6039,579
2,2,6039,2191
3,3,6039,1781
4,4,6039,1839
...,...,...,...
1000204,446,4957,2206
1000205,447,4957,1305
1000206,448,4957,2432
1000207,449,4957,3040


### Split interactions into the train, validation and test datasets using LastNSplitter
We use a widespread splitting strategy Last-One-Out. We filter out cold items and users for simplicity.

In [6]:
from replay.splitters import LastNSplitter

splitter = LastNSplitter(
    N=1,
    divide_column="user_id",
    query_column="user_id",
    strategy="interactions",
    drop_cold_users=True,
    drop_cold_items=True
)

test_events, test_gt = splitter.split(encoded_interactions)
validation_events, validation_gt = splitter.split(test_events)
train_events = validation_events

### Dataset preprocessing ("baking")
SasRec expects each user in the batch to provide their events in a form of a sequence. For this reason, the event splits must be properly processed using the `groupby_sequences` function provided by RePlay.

In [7]:
from replay.data.nn.utils import groupby_sequences


def bake_data(full_data):
    return groupby_sequences(events=full_data, groupby_col="user_id", sort_col="timestamp")

In [8]:
train_events = bake_data(train_events)

validation_events = bake_data(validation_events)
validation_gt = bake_data(validation_gt)

test_events = bake_data(test_events)
test_gt = bake_data(test_gt)

train_events

,user_id,timestamp,item_id
0,0,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[2969, 957, 1574, 1178, 2147, 1658, 3177, 1117..."
1,1,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[1108, 1127, 1120, 2512, 1201, 2735, 1135, 110..."
2,2,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[579, 2651, 1788, 3301, 1781, 1327, 1174, 3429..."
3,3,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[1120, 1025, 466, 3235, 3294, 1106, 1108, 253,..."
4,4,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[847, 2512, 858, 346, 1158, 2651, 2007, 1050, ..."
...,...,...,...
6035,6035,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[1574, 3206, 2183, 1703, 2235, 2651, 2501, 249..."
6036,6036,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[1702, 1848, 1175, 3275, 672, 2932, 802, 548, ..."
6037,6037,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[859, 3165, 1120, 1965, 346, 1288, 1007, 2511,..."
6038,6038,"[0, 1, 2, 3, 4, 5, 6, 7, 8, 9, 10, 11, 12, 13,...","[275, 107, 1886, 869, 1139, 886, 2917, 2809, 2..."


To ensure we don't have unknown users in ground truth, we join validation events and validation ground truth (also join test events and test ground truth correspondingly) by user ids to leave only the common ones.  

In [9]:
def add_gt_to_events(events_df, gt_df):
    gt_to_join = gt_df[["user_id", "item_id"]].rename(columns={"item_id": "ground_truth"})

    events_df = events_df.merge(gt_to_join, on="user_id", how="inner")
    return events_df

validation_events = add_gt_to_events(validation_events, validation_gt)
test_events = add_gt_to_events(test_events, test_gt)

In [10]:
from pathlib import Path

data_dir = Path("temp/data/")
data_dir.mkdir(parents=True, exist_ok=True)

TRAIN_PATH = data_dir / "train.parquet"
VAL_PATH = data_dir / "val.parquet"
PREDICT_PATH = data_dir / "test.parquet"

ENCODER_PATH = data_dir / "encoder"

In [11]:
train_events.to_parquet(TRAIN_PATH)
validation_events.to_parquet(VAL_PATH)
test_events.to_parquet(PREDICT_PATH)

encoder.save(ENCODER_PATH)

/home/jovyan/Pembek/ap_replay/lib/python3.11/site-packages/replay/preprocessing/label_encoder.py:1002: UserWarning: There is already LabelEncoder object saved at the given path. File will be overwrited.
  warnings.warn(msg)
/home/jovyan/Pembek/ap_replay/lib/python3.11/site-packages/replay/preprocessing/label_encoder.py:563: UserWarning: There is already LabelEncodingRule object saved at the given path. File will be overwrited.
  warnings.warn(msg)


# Prepare to model training
### Create the tensor schema
A schema shows the correspondence of columns from the source dataset to the internal representation of tensors inside the model. It is required by the NN models to correctly create embeddings for every source column. Note that user_id is not required in `TensorSchema`.

Note that **cardinality** is the number of unique values ​in the item catalog (vocabulary). **Padding value** is the next value after the last one.

In [12]:
from replay.data import FeatureHint, FeatureType
from replay.data.nn import TensorFeatureInfo, TensorSchema


EMBEDDING_DIM = 64

NUM_UNIQUE_ITEMS = len(encoder.mapping["item_id"])

item_counts = (
    train_events["item_id"]
    .explode()
    .value_counts()
    .reindex(range(NUM_UNIQUE_ITEMS), fill_value=0)
)
item_frequency = torch.tensor((item_counts / item_counts.sum()).to_numpy(), dtype=torch.float32)

tensor_schema = TensorSchema(
    [
        TensorFeatureInfo(
            name="item_id",
            is_seq=True,
            padding_value=NUM_UNIQUE_ITEMS,
            cardinality=NUM_UNIQUE_ITEMS,
            embedding_dim=EMBEDDING_DIM,
            feature_type=FeatureType.CATEGORICAL,
            feature_hint=FeatureHint.ITEM_ID,
        )
    ]
)

### Configure ParquetModule and transformation pipelines

The `ParquetModule` class enables training of models on large datasets by reading data in a batch-wise way. Class is initialized with **paths to every data split, a metadata dict containing shape and padding value information of every column and a dict of transforms**. `ParquetModule`'s  "transform pipelines" are stage-specific modules implementing additional preprocessing to be performed on a batch level right before the forward pass.

For the SasRec model, RePlay provides a function that generates a sequence of appropriate transforms for each data split named **make_default_sasrec_transforms**.

Internally this function creates the following transforms:
1) Training:
    1. Create a target, which contains the shifted item sequence that represents the next item in the sequence (for the next item prediction task).
    2. Rename features to match them with the format expected by the model during training.
    3. Unsqueeze the target (*positive_labels*) and its padding mask (*target_padding_mask*) to get the required shape of these tensors for loss computation.
    4. Group input features to be embedded in the expected format.

2) Validation/Inference:
    1. Rename/group features to match them with the format expected by the model during valdiation/inference.

If a different set of transforms is required, you can create them yourself and submit them to the ParquetModule in the form of a dictionary where the key is the name of the split, and the value is the list of transforms. Available transforms are in the replay/nn/transforms/.

Note: One of the transforms for the training data prepares the initial sequence for the task of the Next Item Prediction so it shifts the sequence of items. For the final sequence length to be correct, you need to set shape of item_id in metadata as **model sequence length + shift**. The default shift value is 1.

In [13]:
from replay.nn.transform.template import make_default_sasrec_transforms

transforms = make_default_sasrec_transforms(tensor_schema)

In [14]:
MAX_SEQ_LEN = 50

train_metadata = {
    "train": {
        "item_id": {"shape": MAX_SEQ_LEN + 1, "padding": tensor_schema["item_id"].padding_value},
    },
    "validate": {
        "item_id": {"shape": MAX_SEQ_LEN, "padding": tensor_schema["item_id"].padding_value},
        "ground_truth": {"shape": 1, "padding": -1}
    },
}

In [15]:
from replay.data.nn import ParquetModule

BATCH_SIZE = 32

parquet_module = ParquetModule(
    train_path=TRAIN_PATH,
    validate_path=VAL_PATH,
    batch_size=BATCH_SIZE,
    metadata=train_metadata,
    transforms=transforms,
)

/tmp/ipykernel_3172512/1761019238.py:5: UserWarning: The following dataset paths aren't provided: test,predict. Make sure to disable these stages in your Lightning Trainer configuration.
  parquet_module = ParquetModule(


## Train model
### Create the SasRec model instance and run the training stage using lightning
We may now train the model using the Lightning trainer class. 

RePlay's implementation of SasRec is designed in a modular, **block-based approach**. Instead of passing configuration parameters to the constructor, SasRec is now built by providing fully initialized components which makes the model more flexible and easier to extend.

In [16]:
from replay.nn.agg import SumAggregator
from replay.nn.embedding import SequenceEmbedding
from replay.nn.loss import PREntmax
from replay.nn.mask import DefaultAttentionMask
from replay.nn.sequential import PositionAwareAggregator, SasRec, SasRecBody, SasRecTransformerLayer

NUM_BLOCKS = 2
NUM_HEADS = 2
DROPOUT = 0.3

excluded_features = list(set([tensor_schema.query_id_feature_name, tensor_schema.timestamp_feature_name]))

body = SasRecBody(
    embedder=SequenceEmbedding(
        schema=tensor_schema,
        categorical_list_feature_aggregation_method="sum",
        excluded_features=excluded_features,
    ),
    embedding_aggregator=PositionAwareAggregator(
        embedding_aggregator=SumAggregator(embedding_dim=EMBEDDING_DIM),
        max_sequence_length=MAX_SEQ_LEN,
        dropout=DROPOUT,
    ),
    attn_mask_builder=DefaultAttentionMask(
        reference_feature_name=tensor_schema.item_id_feature_name,
        num_heads=NUM_HEADS,
    ),
    encoder=SasRecTransformerLayer(
        embedding_dim=EMBEDDING_DIM,
        num_heads=NUM_HEADS,
        num_blocks=NUM_BLOCKS,
        dropout=DROPOUT,
        activation="relu",
        hidden_dim=None,
    ),
    output_normalization=torch.nn.LayerNorm(EMBEDDING_DIM),
)

sasrec = SasRec(
    body=body,
    loss=PREntmax(
        item_frequency=item_frequency,
        ignore_index=tensor_schema.item_id_features.item().padding_value,
    ),
)

A universal PyTorch Lightning module is provided. It can work with any NN model.

In [17]:
from replay.nn.lightning import LightningModule

model = LightningModule(sasrec)

To facilitate training, we add the following callbacks:
1) `ModelCheckpoint` - to save the best trained model based on its Recall metric. It's a default Lightning Callback.
1) `ComputeMetricsCallback` - to display a detailed validation metric matrix after each epoch. It's a custom RePlay callback for computing recsys metrics on validation and test stages.


In [18]:
from lightning.pytorch.callbacks import ModelCheckpoint
from lightning.pytorch.loggers import CSVLogger

from replay.nn.lightning.callback import ComputeMetricsCallback


checkpoint_callback = ModelCheckpoint(
    dirpath="sasrec/checkpoints",
    save_top_k=1,
    verbose=True,
    monitor="recall@10",
    mode="max",
)

validation_metrics_callback = ComputeMetricsCallback(
    metrics=["map", "ndcg", "recall"],
    ks=[1, 5, 10, 20],
    item_count=NUM_UNIQUE_ITEMS,
    verbose=False,
)

csv_logger = CSVLogger(save_dir="sasrec/logs/train", name="SasRec-example")

trainer = L.Trainer(
    max_epochs=5,
    callbacks=[checkpoint_callback, validation_metrics_callback],
    logger=csv_logger,
)

trainer.fit(model, datamodule=parquet_module)

Trainer will use only 1 of 4 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=4)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
You are using a CUDA device ('NVIDIA A100-SXM4-80GB') that has Tensor Cores. To properly utilize them, you should set `torch.set_float32_matmul_precision('medium' | 'high')` which will trade-off precision for performance. For more details, read https://pytorch.org/docs/stable/generated/torch.set_float32_matmul_precision.html#torch.set_float32_matmul_precision
/home/jovyan/Pembek/ap_replay/lib/python3.11/sit

Sanity Checking: |          | 0/? [00:00<?, ?it/s]

/home/jovyan/Pembek/ap_replay/lib/python3.11/site-packages/lightning/pytorch/utilities/data.py:123: Your `IterableDataset` has `__len__` defined. In combination with multi-process data loading (when num_workers > 1), `__len__` could be inaccurate if each worker is not configured independently to avoid having duplicate data.


Training: |          | 0/? [00:00<?, ?it/s]

Validation: |          | 0/? [00:00<?, ?it/s]

...

Epoch 2, global step 567: 'recall@10' reached 0.14605 (best 0.14605), saving model to '/home/jovyan/home/jovyan/Pembek/sasrec/checkpoints/epoch=2-step=567.ckpt' as top 1


Validation: |          | 0/? [00:00<?, ?it/s]

Epoch 3, global step 756: 'recall@10' reached 0.15483 (best 0.15483), saving model to '/home/jovyan/home/jovyan/Pembek/sasrec/checkpoints/epoch=3-step=756.ckpt' as top 1


Validation: |          | 0/? [00:00<?, ?it/s]

Epoch 4, global step 945: 'recall@10' reached 0.16576 (best 0.16576), saving model to '/home/jovyan/home/jovyan/Pembek/sasrec/checkpoints/epoch=4-step=945-v1.ckpt' as top 1
`Trainer.fit` stopped: `max_epochs=5` reached.


Now we can get the best model path and validation metrics from the best epoch.

In [19]:
best_model_path = checkpoint_callback.best_model_path
best_epoch = int(best_model_path.split("epoch=")[1].split("-")[0])
best_epoch_validation_metrics = validation_metrics_callback.get_metrics()[best_epoch]

best_model_path, best_epoch_validation_metrics

('/home/jovyan/home/jovyan/Pembek/sasrec/checkpoints/epoch=4-step=945-v1.ckpt',
 {'recall@1': 0.019208477810025215,
  'ndcg@1': 0.019208477810025215,
  'map@1': 0.019208477810025215,
  'recall@5': 0.09687034040689468,
  'ndcg@5': 0.058033037930727005,
  'map@5': 0.04533587396144867,
  'recall@10': 0.16575591266155243,
  'ndcg@10': 0.08004312217235565,
  'map@10': 0.05426487326622009,
  'recall@20': 0.2732240557670593,
  'ndcg@20': 0.10716070234775543,
  'map@20': 0.06167946010828018})

## Inference

To obtain model scores, we will load the weights from the best checkpoint. To do this, we use the `LightningModule`, providing the path to the checkpoint and the model instance.

In [20]:
body = SasRecBody(
    embedder=SequenceEmbedding(
        schema=tensor_schema,
        categorical_list_feature_aggregation_method="sum",
        excluded_features=excluded_features,
    ),
    embedding_aggregator=PositionAwareAggregator(
        embedding_aggregator=SumAggregator(embedding_dim=EMBEDDING_DIM),
        max_sequence_length=MAX_SEQ_LEN,
        dropout=DROPOUT,
    ),
    attn_mask_builder=DefaultAttentionMask(
        reference_feature_name=tensor_schema.item_id_feature_name,
        num_heads=NUM_HEADS,
    ),
    encoder=SasRecTransformerLayer(
        embedding_dim=EMBEDDING_DIM,
        num_heads=NUM_HEADS,
        num_blocks=NUM_BLOCKS,
        dropout=DROPOUT,
        activation="relu",
        hidden_dim=None,
    ),
    output_normalization=torch.nn.LayerNorm(EMBEDDING_DIM),
)

sasrec = SasRec(
    body=body,
    loss=PREntmax(
        item_frequency=item_frequency,
        ignore_index=tensor_schema.item_id_features.item().padding_value,
    ),
)

best_model = LightningModule.load_from_checkpoint(best_model_path, model=sasrec)
best_model.eval();

Configure `ParquetModule` for inference. It's necessary to read "user_id" column from data for correctly mapping predictions, so add "user_id" to metadata for `ParquetModule`. For reading non-array column dict should be empty.

In [21]:
inference_metadata = {
    "predict": {
        "user_id": {},
        "item_id": {"shape": MAX_SEQ_LEN, "padding": tensor_schema["item_id"].padding_value},
    }
}

parquet_module = ParquetModule(
    predict_path=PREDICT_PATH,
    batch_size=BATCH_SIZE,
    metadata=inference_metadata,
    transforms=transforms,
)

/tmp/ipykernel_3172512/3991935412.py:8: UserWarning: The following dataset paths aren't provided: train,validate,test. Make sure to disable these stages in your Lightning Trainer configuration.
  parquet_module = ParquetModule(


During inference, we can use `TopItemsCallback`. This callback allows getting scores for each user throughout the entire catalog and getting recommendations in the form of ids of items with the highest score values.


Recommendations can be fetched in four formats: PySpark DataFrame, Pandas DataFrame, Polars DataFrame or raw PyTorch tensors. Each of the types corresponds to a callback. In this example, we will use the `PandasTopItemsCallback`.

In [22]:
from replay.nn.lightning.callback import PandasTopItemsCallback

csv_logger = CSVLogger(save_dir="sasrec/logs/test", name="SasRec-example")

TOPK = [1, 5, 10, 20]

pandas_prediction_callback = PandasTopItemsCallback(
    top_k=max(TOPK),
    query_column="user_id",
    item_column="item_id",
    rating_column="score",
)

trainer = L.Trainer(callbacks=[pandas_prediction_callback], logger=csv_logger, inference_mode=True)
trainer.predict(best_model, datamodule=parquet_module, return_predictions=False)

pandas_res = pandas_prediction_callback.get_result()

Trainer will use only 1 of 4 GPUs because it is running inside an interactive / notebook environment. You may try to set `Trainer(devices=4)` but please note that multi-GPU inside interactive / notebook environments is considered experimental and unstable. Your mileage may vary.
GPU available: True (cuda), used: True
TPU available: False, using: 0 TPU cores
💡 Tip: For seamless cloud logging and experiment tracking, try installing [litlogger](https://pypi.org/project/litlogger/) to enable LitLogger, which logs metrics and artifacts automatically to the Lightning Experiments platform.
💡 Tip: For seamless cloud uploads and versioning, try installing [litmodels](https://pypi.org/project/litmodels/) to enable LitModelCheckpoint, which syncs automatically with the Lightning model registry.
LOCAL_RANK: 0 - CUDA_VISIBLE_DEVICES: [0,1,2,3]


Predicting: |          | 0/? [00:00<?, ?it/s]

In [23]:
pandas_res

,user_id,item_id,score
0,0,354,3.805657
0,0,574,3.467723
0,0,740,3.295836
0,0,1727,2.982203
0,0,1900,2.948284
...,...,...,...
6037,6039,1259,1.085238
6037,6039,440,1.077175
6037,6039,2884,1.070007
6037,6039,2099,1.054049


### Calculating metrics

*test_gt* is already encoded, so we can use it for computing metrics.

In [24]:
from replay.metrics import MAP, OfflineMetrics, Precision, Recall
from replay.metrics.torch_metrics_builder import metrics_to_df

In [25]:
result_metrics = OfflineMetrics(
    [Recall(TOPK), Precision(TOPK), MAP(TOPK)],
    query_column="user_id",
    rating_column="score",
)(pandas_res, test_gt.explode("item_id"))

In [26]:
metrics_to_df(result_metrics)

k,1,5,10,20
MAP,0.020537,0.046922,0.055442,0.061531
Precision,0.020537,0.019874,0.016363,0.012695
Recall,0.020537,0.099371,0.163630,0.253892


Let's call the `inverse_transform` encoder's function to get the final dataframe with recommendations

In [27]:
encoder = LabelEncoder.load(ENCODER_PATH)
encoder.inverse_transform(pandas_res)

,user_id,item_id,score
0,1,364,3.805657
0,1,588,3.467723
0,1,783,3.295836
0,1,1907,2.982203
0,1,2081,2.948284
...,...,...,...
6037,6040,1357,1.085238
6037,6040,454,1.077175
6037,6040,3100,1.070007
6037,6040,2291,1.054049
